### Firewall API - Administration Management

[![Python 3.11+](https://img.shields.io/badge/Python-3.11%2B-blue.svg)](https://www.python.org/downloads/)
[![License: MIT](https://img.shields.io/badge/License-MIT-yellow.svg)](../LICENSE)
[![Development Status: Beta 0.8.0](https://img.shields.io/badge/Status-Beta%200.8.0-orange.svg)](https://pypi.org/project/sfos-api/)
[![Code Style: Ruff](https://img.shields.io/endpoint?url=https://raw.githubusercontent.com/astral-sh/ruff/main/assets/badge/v2.json)](https://github.com/astral-sh/ruff)

**Purpose:** Create, read, update, and delete administration resources including schedules, access profiles, role-based access permissions, system messages, and singleton system, API, and portal settings.

**Topics:**
- Basic CRUD operations using domain StrEnum objects (`Administration`)
- Recurring and one-time schedule management (`Administration.SCHEDULES`)
- Administrator access profiles and role permissions (`Administration.ACCESS_PROFILES`, `Administration.ACCESS_PERMISSIONS`)
- System settings, API access configuration, and portal settings (`Administration.SYSTEM_SETTINGS`, `Administration.API_SETTINGS`, `Administration.PORTAL_SETTINGS`)
- Support access and system message templates (`Administration.SUPPORT_ACCESS`, `Administration.MESSAGES`)
- Filtering and auto-pagination
- Error handling
- Bulk operations

**Requirements:**
- Credentials configured (`.credentials/config.ini`)
- SFOS firewall accessible
- API key with appropriate permissions

---

#### Setup: Import Libraries and Configure Client

In [ ]:
from utils_api import find_credentials_file, print_data

from sfos_api import Administration, SFOSClient

# Initialize the SFOS API Client
client = SFOSClient(config_path=find_credentials_file())
print("✅ SFOS API client initialized")

#### Create Schedules

Create schedule objects using the `Administration.SCHEDULES` endpoint object.

Schedules define time periods (recurring or one-time) used in firewall rules and security policies.

API Link: [Create a schedule | Sophos Firewall API](https://docs.sophos.com/nsg/sfos-api/rest-api/api/firewall-config-openapi/create-schedule/)

---


In [ ]:
# Create a recurring schedule (business hours)
endpoint = Administration.SCHEDULES
schedule_name = "test_schedule_work_hours"

payload = {
    "name": schedule_name,
    "description": "Business hours recurring schedule",
    "type": "recurring",
    "timeSlots": [
        {
            "dayOfWeek": "weekdays",
            "startTime": "09:00",
            "endTime": "17:00",
        },
    ],
}

try:
    response = client.post(endpoint, payload)
    print(f"✅ Created recurring schedule '{schedule_name}'")
    print_data(response)
except Exception as e:
    print(f"Error creating schedule: {e}")
    print_data(str(e))

In [ ]:
# Create a one-time schedule (maintenance window)
endpoint = Administration.SCHEDULES
schedule_name = "test_schedule_maintenance"

payload = {
    "name": schedule_name,
    "description": "One-time system maintenance window",
    "type": "oneTime",
    "startDate": "2026-11-01 00:00:00",
    "endDate": "2026-11-01 04:00:00",
    "timeSlots": [
        {
            "dayOfWeek": "allDays",
            "startTime": "00:00",
            "endTime": "04:00",
        },
    ],
}

try:
    response = client.post(endpoint, payload)
    print(f"✅ Created one-time schedule '{schedule_name}'")
    print_data(response)
except Exception as e:
    print(f"Error creating schedule: {e}")
    print_data(str(e))

#### Read Schedules

Retrieve schedule information by ID/name or list all schedules.

API Links: [Get all schedules | Sophos Firewall API](https://docs.sophos.com/nsg/sfos-api/rest-api/api/firewall-config-openapi/list-schedules/) | [Get a schedule | Sophos Firewall API](https://docs.sophos.com/nsg/sfos-api/rest-api/api/firewall-config-openapi/get-schedule/)

---


In [ ]:
# Get a specific schedule by name
endpoint = Administration.SCHEDULES
schedule_name = "test_schedule_work_hours"

try:
    response = client.get(endpoint, id_or_name=schedule_name)
    print(f"✅ Retrieved schedule '{schedule_name}'")
    print_data(response)
except Exception as e:
    print(f"Error reading schedule: {e}")
    print_data(str(e))

In [ ]:
# Get all schedules
endpoint = Administration.SCHEDULES

try:
    response = client.get(endpoint)

    if "items" in response and response["items"]:
        print(f"✅ Retrieved {len(response['items'])} schedules")
        print_data(response["items"], include=["name", "description", "type"])
    else:
        print("No schedules found")
except Exception as e:
    print(f"Error reading schedules: {e}")
    print_data(str(e))

In [ ]:
# Get schedules with filtering
endpoint = Administration.SCHEDULES
params = {
    "nameContains": "test",
}

try:
    response = client.get(endpoint, params=params)

    if "items" in response:
        print(f"✅ Retrieved {len(response['items'])} schedules containing 'test'")
        print_data(response["items"], include=["name", "description", "type", "timeSlots"])
    else:
        print("No schedules found matching filter")
except Exception as e:
    print(f"Error reading schedules: {e}")
    print_data(str(e))

#### Update Schedule

Modify an existing schedule's properties.

API Link: [Update a schedule | Sophos Firewall API](https://docs.sophos.com/nsg/sfos-api/rest-api/api/firewall-config-openapi/update-schedule/)

---


In [ ]:
# Update a schedule
endpoint = Administration.SCHEDULES
schedule_name = "test_schedule_work_hours"

payload = {
    "description": "Updated business hours schedule (08:30 - 17:30)",
    "timeSlots": [
        {
            "dayOfWeek": "weekdays",
            "startTime": "08:30",
            "endTime": "17:30",
        },
    ],
}

try:
    response = client.patch(endpoint, id_or_name=schedule_name, data=payload)
    print(f"✅ Updated schedule '{schedule_name}'")
    print_data(response)
except Exception as e:
    print(f"Error updating schedule: {e}")
    print_data(str(e))

#### Delete Schedules

Remove schedule objects (single or bulk delete).

API Links: [Delete a schedule | Sophos Firewall API](https://docs.sophos.com/nsg/sfos-api/rest-api/api/firewall-config-openapi/delete-schedule/) | [Bulk delete schedules | Sophos Firewall API](https://docs.sophos.com/nsg/sfos-api/rest-api/api/firewall-config-openapi/bulk-delete-schedules/)

---


In [ ]:
# Delete a single schedule
endpoint = Administration.SCHEDULES
schedule_name = "test_schedule_maintenance"

try:
    response = client.delete(endpoint, id_or_name=schedule_name)
    print(f"✅ Deleted schedule '{schedule_name}'")
    if response:
        print_data(response)
except Exception as e:
    print(f"Error deleting schedule: {e}")
    print_data(str(e))

In [ ]:
# Bulk delete schedules
endpoint = Administration.SCHEDULES

items_to_delete = [
    {"name": "test_schedule_work_hours"},
]

try:
    response = client.delete(endpoint, items=items_to_delete)
    print(f"✅ Bulk deleted {len(items_to_delete)} schedules")
    if response:
        print_data(response)
except Exception as e:
    print(f"Error bulk deleting schedules: {e}")
    print_data(str(e))

In [ ]:
# Bulk delete schedules with filter
endpoint = Administration.SCHEDULES

try:
    response = client.delete(endpoint, nameContains="test_")
    print("✅ Bulk deleted schedules matching nameContains='test_'")
    if response:
        print_data(response)
except Exception as e:
    print(f"Error bulk deleting schedules: {e}")
    print_data(str(e))

#### Access Permissions and Administrator Profiles

Manage administrator role profiles and inspect available granular permissions using `Administration.ACCESS_PERMISSIONS` and `Administration.ACCESS_PROFILES`.

API Links: [Get all permissions | Sophos Firewall API](https://docs.sophos.com/nsg/sfos-api/rest-api/api/firewall-config-openapi/get-all-permissions/) | [Get all access profiles | Sophos Firewall API](https://docs.sophos.com/nsg/sfos-api/rest-api/api/firewall-config-openapi/get-all-access-profiles/) | [Create access profile | Sophos Firewall API](https://docs.sophos.com/nsg/sfos-api/rest-api/api/firewall-config-openapi/create-access-profile/)

---


In [ ]:
# Get all granular administrator permissions
endpoint = Administration.ACCESS_PERMISSIONS

try:
    response = client.get(endpoint)
    if "items" in response and response["items"]:
        print(f"✅ Retrieved {len(response['items'])} administrator permissions")
        print_data(response["items"], include=["name", "displayName", "supportedAccessLevels"])
    else:
        print_data(response)
except Exception as e:
    print(f"Error reading access permissions: {e}")
    print_data(str(e))

In [ ]:
# Create an administrator access profile
endpoint = Administration.ACCESS_PROFILES
profile_name = "test_profile_auditor"

payload = {
    "name": profile_name,
    "permissions": [
        {"name": "networkAndRouting", "access": "readOnly"},
        {"name": "rulesPoliciesThreatSettings", "access": "readOnly"},
        {"name": "logSettings", "access": "readOnly"},
        {"name": "reports", "access": "readOnly"},
    ],
}

try:
    response = client.post(endpoint, payload)
    print(f"✅ Created access profile '{profile_name}'")
    print_data(response)
except Exception as e:
    print(f"Error creating access profile: {e}")
    print_data(str(e))

In [ ]:
# Get an access profile by name
endpoint = Administration.ACCESS_PROFILES
profile_name = "test_profile_auditor"

try:
    response = client.get(endpoint, id_or_name=profile_name)
    print(f"✅ Retrieved access profile '{profile_name}'")
    print_data(response)
except Exception as e:
    print(f"Error reading access profile: {e}")
    print_data(str(e))

In [ ]:
# Get all access profiles
endpoint = Administration.ACCESS_PROFILES

try:
    response = client.get(endpoint)
    if "items" in response and response["items"]:
        print(f"✅ Retrieved {len(response['items'])} access profiles")
        print_data(response["items"], include=["name", "permissions"])
    else:
        print("No access profiles found")
except Exception as e:
    print(f"Error reading access profiles: {e}")
    print_data(str(e))

In [ ]:
# Update an access profile
endpoint = Administration.ACCESS_PROFILES
profile_name = "test_profile_auditor"

payload = {
    "permissions": [
        {"name": "networkAndRouting", "access": "readOnly"},
        {"name": "rulesPoliciesThreatSettings", "access": "readOnly"},
        {"name": "logSettings", "access": "readOnly"},
        {"name": "reports", "access": "readOnly"},
        {"name": "vpnTunnelManagement", "access": "readOnly"},
    ],
}

try:
    response = client.patch(endpoint, id_or_name=profile_name, data=payload)
    print(f"✅ Updated access profile '{profile_name}'")
    print_data(response)
except Exception as e:
    print(f"Error updating access profile: {e}")
    print_data(str(e))

In [ ]:
# Delete an access profile
endpoint = Administration.ACCESS_PROFILES
profile_name = "test_profile_auditor"

try:
    response = client.delete(endpoint, id_or_name=profile_name)
    print(f"✅ Deleted access profile '{profile_name}'")
    if response:
        print_data(response)
except Exception as e:
    print(f"Error deleting access profile: {e}")
    print_data(str(e))

#### System & Service Configurations

Query and inspect singleton administration configurations:
- `Administration.SYSTEM_SETTINGS`: Hostname and login disclaimer
- `Administration.API_SETTINGS`: API access status and allowed IP hosts
- `Administration.PORTAL_SETTINGS`: WebAdmin, User, and VPN portal ports and certificates
- `Administration.SUPPORT_ACCESS`: Secure support tunnel status and duration
- `Administration.MESSAGES`: Custom login, disclaimer, and notification banners

API Links: [Get system settings | Sophos Firewall API](https://docs.sophos.com/nsg/sfos-api/rest-api/api/firewall-config-openapi/get-system-settings/) | [Get API settings | Sophos Firewall API](https://docs.sophos.com/nsg/sfos-api/rest-api/api/firewall-config-openapi/get-api-settings/) | [Get portal settings | Sophos Firewall API](https://docs.sophos.com/nsg/sfos-api/rest-api/api/firewall-config-openapi/get-portal-settings/)

---


In [ ]:
# Read system settings (hostname, login disclaimer)
endpoint = Administration.SYSTEM_SETTINGS

try:
    response = client.get(endpoint)
    print("✅ Retrieved system settings")
    print_data(response)
except Exception as e:
    print(f"Error reading system settings: {e}")
    print_data(str(e))

In [ ]:
# Read API access settings and allowed hosts
endpoint = Administration.API_SETTINGS

try:
    response = client.get(endpoint)
    print("✅ Retrieved API settings")
    print_data(response)
except Exception as e:
    print(f"Error reading API settings: {e}")
    print_data(str(e))

In [ ]:
# Read portal settings (ports and certificates for WebAdmin/User/VPN portals)
endpoint = Administration.PORTAL_SETTINGS

try:
    response = client.get(endpoint)
    print("✅ Retrieved portal settings")
    print_data(response)
except Exception as e:
    print(f"Error reading portal settings: {e}")
    print_data(str(e))

In [ ]:
# Read technical support access settings
endpoint = Administration.SUPPORT_ACCESS

try:
    response = client.get(endpoint)
    print("✅ Retrieved support access settings")
    print_data(response)
except Exception as e:
    print(f"Error reading support access: {e}")
    print_data(str(e))

In [ ]:
# Read administration login messages and legal banners
endpoint = Administration.MESSAGES

try:
    response = client.get(endpoint)
    print("✅ Retrieved administration messages")
    if "messages" in response and response["messages"]:
        print(f"Retrieved {len(response['messages'])} configured message templates")
        print_data(response["messages"][:3], include=["name", "displayName", "group"])
    else:
        print_data(response)
except Exception as e:
    print(f"Error reading administration messages: {e}")
    print_data(str(e))

In [ ]:
# Close the client session when done
client.close()
print("✅ Client session closed")